<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/06_foundation_models/notebooks/sesion_06_6_integrador_estilo_llm.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/06_foundation_models/notebooks/sesion_06_6_integrador_estilo_llm.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F06_foundation_models%2Fnotebooks%2Fsesion_06_6_integrador_estilo_llm.ipynb)

# SI7011 · Sesión 06 · Ejercicio integrador: ¿hay que ajustar un LLM para que responda con un estilo?
**Juan David Martínez Vargas · Universidad EAFIT**

En el notebook 6.3 convertiste un modelo **base** en uno que responde. Aquí partimos de un modelo que **ya conversa**, Qwen2.5-0.5B-Instruct, y queremos que responda en el castellano de *Don Quijote*. La pregunta: **¿basta con pedírselo en el prompt, o hay que ajustarlo?** Lo haces como en la práctica: datos en el formato de chat de TRL, `SFTTrainer` con LoRA de peft, y MLflow.

| Etapa | Qué haces | Artefacto |
|---|---|---|
| Datos | unas 380 preguntas de Dolly con su respuesta reescrita en el estilo: 40 para validar, unas 60 de test y el resto para entrenar (listo) | ficha de datos en MLflow |
| Formato | cada fila como conversación *prompt / completion* (**TODO 1**) | |
| Ajuste | `LoraConfig` y `SFTConfig` (**TODO 2**); `SFTTrainer` registra en MLflow (listo) | `lora_estilo/` |
| Evaluación | $J$ de test para base, base con el estilo en el prompt y LoRA (**TODO 3**); un juez simple del estilo (listo) | tablas en MLflow |
| Despliegue | `responder_raw`: de una pregunta en texto a una respuesta, con contrato (**TODO 4**) | |

**Usa GPU** (Kaggle T4, unos 10 minutos) con **Internet** activado (el modelo viene de Hugging Face y TRL se instala con `pip`). **Datos:** en Kaggle agrega el dataset [`juanmartinezv4399/si7011-s06-dolly-quijote`](https://www.kaggle.com/datasets/juanmartinezv4399/si7011-s06-dolly-quijote) con **Add Input**; en Colab se baja con tu secreto `KAGGLE_API_TOKEN`.

<span style="font-size:smaller">Adaptado de MIT 6.S191 *Introduction to Deep Learning*, Lab 3 (Copyright (c) 2025 MIT Introduction to Deep Learning, licencia MIT). Las preguntas son de Databricks Dolly-15k en español (CC BY-SA 3.0); las respuestas con estilo se generaron con Qwen2.5-7B-Instruct.</span>

In [1]:
%pip install -q mlflow "trl==1.16.0" peft

In [2]:
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')     # Kaggle "GPU T4 x2": una sola GPU (SFTTrainer no admite DataParallel)
# Kaggle trae torchao 0.10, que las versiones recientes de transformers y peft rechazan al importarse
# ("Found an incompatible version of torchao"). Este notebook no lo usa: si es viejo, se desinstala.
import os
from importlib.metadata import version, PackageNotFoundError
try:
    if tuple(int(x) for x in version('torchao').split('.')[:2]) < (0, 16):
        os.system('pip uninstall -y -q torchao')
except (PackageNotFoundError, ValueError):
    pass

import hashlib, json, math, random, textwrap, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers, trl, peft
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer

os.environ.setdefault('MLFLOW_DISABLE_AGENT_HINT', '1')
import mlflow
mlflow.set_tracking_uri('sqlite:///mlflow.db')
os.environ['MLFLOW_TRACKING_URI'] = 'sqlite:///mlflow.db'       # el que usa el Trainer de Hugging Face
os.environ['MLFLOW_EXPERIMENT_NAME'] = 'S06-estilo-llm'
mlflow.set_experiment('S06-estilo-llm')

random.seed(0); torch.manual_seed(0)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '· trl', trl.__version__, '· peft', peft.__version__,
      '· MLflow', mlflow.__version__, '·', dev)

2026/11/02 04:41:23 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/02 04:41:23 INFO mlflow.store.db.utils: Updating database tables
2026/11/02 04:41:25 INFO mlflow.tracking.fluent: Experiment with name 'S06-estilo-llm' does not exist. Creating a new experiment.


transformers 5.21.0 · trl 1.16.0 · peft 0.21.2 · MLflow 3.17.2 · cuda


## 1. Datos (listo)
Cada fila: una pregunta de Dolly en español, la respuesta original (`response`) y la misma respuesta reescrita en el estilo (`response_estilo`). Son preguntas **distintas** de las del notebook 6.3. Del `train` separamos 40 para validar, y de `test` quitamos las preguntas que también están en `train`; `test` se usa solo en el TODO 3. La última línea mide cuántas respuestas de entrenamiento usan algún giro de la época: lo que el modelo tiene para aprender.

In [3]:
KAGGLE_HANDLE = 'juanmartinezv4399/si7011-s06-dolly-quijote'
ROOTS = [Path('.'), Path('data'), Path('/kaggle/input'), Path('/teamspace/studios/this_studio/data')]

def kaggle_token():
    # Colab: secreto KAGGLE_API_TOKEN. Local o Lightning: variable de entorno o ~/.kaggle/access_token.
    if 'KAGGLE_API_TOKEN' not in os.environ:
        try:
            from google.colab import userdata
            os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN').strip()
        except Exception:
            pass
    return 'KAGGLE_API_TOKEN' in os.environ or (Path.home() / '.kaggle' / 'access_token').exists()

def buscar(roots, patron='dolly_quijote_es.csv'):
    for r in [r for r in roots if r.exists()]:
        hits = sorted(r.rglob(patron))
        if hits:
            return hits[0]
    return None

ruta = buscar(ROOTS)
if ruta is None and kaggle_token():
    import kagglehub                                        # preinstalado en Colab y Kaggle
    ruta = buscar([Path(kagglehub.dataset_download(KAGGLE_HANDLE))])
assert ruta is not None, f'No encontré dolly_quijote_es.csv: en Kaggle agrega {KAGGLE_HANDLE} con Add Input.'
df = pd.read_csv(ruta)
train_all, test = df[df.split == 'train'].reset_index(drop=True), df[df.split == 'test'].reset_index(drop=True)
test = test[~test.instruction.isin(train_all.instruction)].reset_index(drop=True)   # ninguna pregunta de test se vio al entrenar
val, train = train_all.iloc[:40].reset_index(drop=True), train_all.iloc[40:].reset_index(drop=True)
if dev == 'cpu':
    train, val, test = train.iloc[:40], val.iloc[:12], test.iloc[:12]
data_card = {'fuente': str(ruta), 'sha256': hashlib.sha256(ruta.read_bytes()).hexdigest()[:16],
             'train': len(train), 'val': len(val), 'test': len(test)}
print(data_card)
GIROS = ['vuestra merced', 'agora', 'mesmo', 'asaz', 'os digo', 'pardiez', 'aqueste', 'hube de', 'sabed', 'en verdad']
con_giros = lambda r: sum(g in r.lower() for g in GIROS) >= 1      # un juez muy simple del estilo
print(f"respuestas de entrenamiento con algún giro de la época: {train.response_estilo.apply(con_giros).mean():.0%} "
      f"(Dolly original: {train.response.apply(con_giros).mean():.0%})")
e = train.iloc[0]
print('\n▶', e.instruction, '\n  Dolly:  ', e.response, '\n  estilo: ', e.response_estilo)

100%|██████████| 93.9k/93.9k [00:00<00:00, 48.8MB/s]

Extracting files...
{'fuente': '/root/.cache/kagglehub/datasets/juanmartinezv4399/si7011-s06-dolly-quijote/versions/2/dolly_quijote_es.csv', 'sha256': '7f7508c7505180f6', 'train': 284, 'val': 40, 'test': 59}
respuestas de entrenamiento con algún giro de la época: 100% (Dolly original: 0%)

▶ ¿Cuáles son los elementos típicos utilizados para crear un Haft-sin? 
  Dolly:   Ajo, manzanas, monedas, brotes de trigo, brotes de cebada, brotes de lentejas, pudin dulce de germen de trigo, frutos secos de oleastro, zumaque, vinagre. 
  estilo:  Ajo, manzanas, monedas, brotes de trigo, brotes de cebada, brotes de lentejas, pudín dulce de germen de trigo, frutos secos de oleastro, zumaque, y vinagre, son los elementos típicos que se usan para crear un Haft-sin, como os digo agora mismo.


## 2. El modelo (listo)
`mensajes` arma la conversación (con o sin mensaje de sistema) y `responder` aplica la **plantilla de chat del modelo** (`apply_chat_template`) y genera con greedy hasta `<|im_end|>`. Mira la plantilla, y cómo responde el modelo con y sin el estilo en el sistema.

In [4]:
MODELO = 'Qwen/Qwen2.5-0.5B-Instruct'      # la versión que ya conversa del modelo base del notebook 6.3
tok = AutoTokenizer.from_pretrained(MODELO)
modelo = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float32).to(dev)
FIN = tok.convert_tokens_to_ids('<|im_end|>')               # el fin de turno de la plantilla de Qwen
ESTILO = ('Responde siempre en el castellano de Don Quijote de la Mancha (principios del siglo XVII): '
          'vuestra merced, agora, mesmo, asaz, os digo; tono cortés y algo grandilocuente.')

def mensajes(pregunta, sistema=None):
    return ([{'role': 'system', 'content': sistema}] if sistema else []) + [{'role': 'user', 'content': pregunta}]

@torch.no_grad()
def responder(m, pregunta, sistema=None, max_nuevos=150):
    x = tok.apply_chat_template(mensajes(pregunta, sistema), add_generation_prompt=True, return_tensors='pt',
                                return_dict=True).to(dev)
    out = m.generate(**x, max_new_tokens=max_nuevos, do_sample=False, eos_token_id=FIN,
                     pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0, x['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print(tok.apply_chat_template(mensajes('¿Qué es un átomo?'), add_generation_prompt=True, tokenize=False))
for q in test.instruction[:2]:
    print('▶', q)
    print('  base:          ', textwrap.shorten(responder(modelo, q), 300))
    print('  base + estilo: ', textwrap.shorten(responder(modelo, q, ESTILO), 300))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
¿Qué es un átomo?<|im_end|>
<|im_start|>assistant

▶ ¿Qué es la bicicleta de gravilla?
  base:           La bicicleta de gravilla es una forma de bicicleta que se utiliza principalmente en el mundo del deporte y la recreación. Aquí hay algunos puntos clave sobre esta bicicleta: 1. Desarrollo: La bicicleta de gravilla fue desarrollada por el ingeniero italiano Luigi Galvani en 1809. 2. Propósito [...]
  base + estilo:  La bicicleta de gravilla es una bicicleta diseñada para ser usada como un vehículo de transporte por tierra. Es conocida por su diseño simple y eficiente, con una sola rueda trasera que permite una velocidad muy rápida. Esta bicicleta se utiliza principalmente para transportar personas o [...]
▶ Cuéntame un chiste sobre un ave originaria de la Antártida.
  base:           Aquí tienes un chiste sobre una ave originaria de la Antártida: ¿Qué hace una ave en la 

## 3. TODO 1 · Cada fila como conversación · 5 min
TRL entrena con conversaciones en el formato *prompt / completion*: el prompt es lo que el modelo lee, la completion lo que tiene que aprender a escribir. Escribe `a_conversacion(fila, sistema=None)` que devuelva

```python
{'prompt': mensajes(pregunta, sistema),
 'completion': [{'role': 'assistant', 'content': <la respuesta CON estilo>}]}
```

`conjunto` (listo) convierte una tabla entera en un `Dataset` de Hugging Face.

In [5]:
def a_conversacion(fila, sistema=None):
    # TODO 1: el prompt es lo que el modelo lee (sistema opcional + pregunta del usuario);
    # la completion es lo que debe aprender a escribir: la respuesta CON estilo.
    return {
        'prompt': mensajes(fila['instruction'], sistema),
        'completion': [{'role': 'assistant', 'content': fila['response_estilo']}],
    }

In [6]:
def conjunto(filas, sistema=None):
    """Un Dataset de Hugging Face en el formato prompt / completion conversacional de TRL."""
    return Dataset.from_list([a_conversacion(f, sistema) for f in filas.to_dict('records')])

ej = a_conversacion(train.iloc[0].to_dict())
assert set(ej) == {'prompt', 'completion'}, "Esperaba las claves 'prompt' y 'completion'."
assert ej['prompt'][-1] == {'role': 'user', 'content': train.instruction[0]}, 'El prompt termina con el mensaje del usuario.'
assert ej['completion'] == [{'role': 'assistant', 'content': train.response_estilo[0]}], \
    'La completion es un solo mensaje del asistente con la respuesta CON estilo.'
ej_s = a_conversacion(train.iloc[0].to_dict(), ESTILO)
assert ej_s['prompt'][0] == {'role': 'system', 'content': ESTILO} and len(ej_s['prompt']) == 2, \
    'Con sistema, el prompt empieza con el mensaje de sistema.'
D_train, D_val = conjunto(train), conjunto(val)
print('a_conversacion correcta ·', len(D_train), 'conversaciones de entrenamiento')
print(D_train[0])

a_conversacion correcta · 284 conversaciones de entrenamiento
{'prompt': [{'content': '¿Cuáles son los elementos típicos utilizados para crear un Haft-sin?', 'role': 'user'}], 'completion': [{'content': 'Ajo, manzanas, monedas, brotes de trigo, brotes de cebada, brotes de lentejas, pudín dulce de germen de trigo, frutos secos de oleastro, zumaque, y vinagre, son los elementos típicos que se usan para crear un Haft-sin, como os digo agora mismo.', 'role': 'assistant'}]}


## 4. TODO 2 · LoRA y el entrenamiento supervisado · 10 min
Define dos objetos (mira el notebook 6.3, sección 5):
- `lora = LoraConfig(...)`: $r = 16$, `lora_alpha=32`, `lora_dropout=0.0`, `task_type='CAUSAL_LM'`, sobre `q_proj`, `k_proj`, `v_proj`, `o_proj`;
- `args = SFTConfig(...)`: `output_dir='sft_estilo'`, `run_name='1-lora-r16'`, la pérdida **solo en la respuesta** (`completion_only_loss=True`), `max_length=384`, dos épocas, lotes de 8 (entrenamiento y evaluación), `learning_rate=5e-4`, coseno con `warmup_steps=0.1`, `fp16` en la GPU y `bf16=False` (la T4 no tiene bfloat16), evaluación cada 10 pasos (`eval_strategy='steps'`, `eval_steps=10`), `logging_steps=5`, `save_strategy='no'` y `report_to='mlflow'`.

La celda de prueba crea el `SFTTrainer` y te muestra qué tokens cuentan en la pérdida: deben ser solo la respuesta y el fin de turno.

**Predice:** en el TODO 3 vas a medir $J$ sobre las respuestas con estilo de test en tres condiciones. ¿La del modelo con el estilo **pedido en el prompt** quedará más cerca de la del modelo base o de la de LoRA?

In [7]:
# TODO 2: LoRA y el entrenamiento supervisado
# LoRA: en vez de ajustar los ~500 M de pesos, se entrenan dos matrices pequeñas (A y B, de rango r)
# en las cuatro proyecciones de la atención; el modelo original queda congelado.
lora = LoraConfig(
    r=16,                 # rango de las matrices A y B
    lora_alpha=32,        # escala del cambio: alpha / r = 2
    lora_dropout=0.0,
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

args = SFTConfig(
    output_dir='sft_estilo',
    run_name='1-lora-r16',
    completion_only_loss=True,          # la pérdida cuenta solo la respuesta, no la pregunta
    max_length=384,
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-4,
    lr_scheduler_type='cosine',
    warmup_steps=0.1,                   # 10 % de los pasos de warmup
    fp16=dev == 'cuda',                 # la T4 no tiene bfloat16
    bf16=False,
    eval_strategy='steps',
    eval_steps=10,
    logging_steps=5,
    save_strategy='no',
    report_to='mlflow',
)

In [8]:
assert lora is not None and args is not None, 'Completa TODO 2: define lora y args.'
assert lora.r == 16 and set(lora.target_modules) == {'q_proj', 'k_proj', 'v_proj', 'o_proj'}, \
    'LoRA con r = 16 en las cuatro matrices de la atención.'
assert args.completion_only_loss is True, 'La pérdida debe contar solo la respuesta (completion_only_loss).'
assert args.num_train_epochs == 2 and args.per_device_train_batch_size == 8 and args.learning_rate == 5e-4
assert args.eval_strategy == 'steps' and 'mlflow' in args.report_to, 'Evalúa cada pocos pasos y registra en MLflow.'
assert not args.bf16, 'La T4 no tiene bfloat16: usa fp16.'
trainer = SFTTrainer(model=modelo, args=args, train_dataset=D_train, eval_dataset=D_val,
                     processing_class=tok, peft_config=lora)
trainer.model.print_trainable_parameters()
lote = next(iter(trainer.get_train_dataloader()))
ids, etiquetas = lote['input_ids'][0], lote['labels'][0]
print('cuenta en la pérdida:', repr(tok.decode(ids[etiquetas != -100])))

Tokenizing train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359
cuenta en la pérdida: 'Aqueste es el gran diferencial entre Spark y MapReduce: mientras que MapReduce opera y guarda los datos en disco para cada paso, hube de decir que Spark los conserva en memoria para los pasos subsiguientes, lo que le confiere una notable agilidad y rapidez.<|im_end|>\n'


## 5. Entrenar (listo)
`trainer.train()` registra en MLflow los hiperparámetros, la pérdida de entrenamiento y la de validación. Después guardamos **solo el adaptador** (las matrices $A_r$ y $B$, unos MB) con el **contrato**, y los subimos a la misma corrida.

In [9]:
t0 = time.time()
trainer.train()
print(f'{time.time() - t0:.0f} s')
run_id = mlflow.last_active_run().info.run_id              # la corrida que abrió el Trainer
CONTRATO = {'modelo': MODELO, 'max_tokens_pregunta': 128, 'max_nuevos': 150, 'plantilla': 'chat de Qwen, sin sistema'}
trainer.model.save_pretrained('lora_estilo')               # solo el adaptador: unos MB
json.dump(CONTRATO, open('lora_estilo/contrato.json', 'w'), ensure_ascii=False, indent=1)
with mlflow.start_run(run_id=run_id):
    mlflow.log_params({f'data_{k}': v for k, v in data_card.items()})
    mlflow.log_artifacts('lora_estilo', 'lora_estilo')
print('adaptador y contrato en MLflow, corrida', run_id)

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,2.352406,2.382675,2.587878,0.521827,12893.000000
20,2.404778,2.299743,2.520117,0.530764,25677.000000
30,2.254022,2.166678,2.198546,0.550737,37567.000000
40,1.979202,2.137199,2.140187,0.558745,49544.000000
50,1.959438,2.120269,2.075849,0.558703,62739.000000
60,1.939278,2.108640,2.090354,0.559873,74944.000000
70,1.945174,2.106371,2.094411,0.559145,87241.000000
72,1.945174,2.106489,2.094467,0.559868,89634.000000


47 s
adaptador y contrato en MLflow, corrida 933db7563f8242aea9756f751ffd5aa4


In [10]:
def nll(datos):
    """La J de la respuesta sobre un Dataset, con el modelo como esté (con o sin el adaptador activo)."""
    ev = SFTTrainer(model=trainer.model, processing_class=tok, train_dataset=datos, eval_dataset=datos,
                    args=SFTConfig(output_dir='eval_tmp', per_device_eval_batch_size=8, max_length=384,
                                   fp16=dev == 'cuda', bf16=False, report_to='none'))
    return ev.evaluate()['eval_loss']

## 6. TODO 3 · ¿Basta el prompt? · 15 min
Escribe `comparar(filas)`: devuelve un diccionario con la $J$ de las respuestas con estilo de `filas` en tres condiciones (diapositiva 74):

| Clave | Adaptador | Mensaje de sistema |
|---|---|---|
| `'base'` | apagado (`with trainer.model.disable_adapter():`) | ninguno |
| `'base + estilo en el prompt'` | apagado | `ESTILO` |
| `'LoRA'` | activo | ninguno |

`nll(conjunto(filas, sistema))` ya calcula $J$ sobre un conjunto. La comparación es justa porque las tres condiciones usan el **mismo tokenizador** y la misma respuesta objetivo: solo cambia el prompt o los pesos.

In [11]:
def comparar(filas):
    # TODO 3: la J de las mismas respuestas con estilo en tres condiciones.
    # Solo cambia el prompt (con o sin ESTILO) o los pesos (adaptador apagado o activo).
    r = {}
    with trainer.model.disable_adapter():                          # adaptador apagado = modelo base
        r['base'] = nll(conjunto(filas))                           # sin sistema
        r['base + estilo en el prompt'] = nll(conjunto(filas, ESTILO))  # el estilo pedido en el sistema
    r['LoRA'] = nll(conjunto(filas))                               # adaptador activo, sin sistema
    return r

In [12]:
r = comparar(test.iloc[:8])
assert set(r) == {'base', 'base + estilo en el prompt', 'LoRA'}, 'Esperaba las tres condiciones.'
with trainer.model.disable_adapter():
    assert abs(r['base'] - nll(conjunto(test.iloc[:8]))) < 1e-3, '"base" se mide con el adaptador apagado y sin sistema.'
assert abs(r['LoRA'] - nll(conjunto(test.iloc[:8]))) < 1e-3, '"LoRA" se mide con el adaptador activo y sin sistema.'
assert r['base + estilo en el prompt'] != r['base'], 'La condición con el estilo en el prompt usa sistema=ESTILO.'

tabla = pd.DataFrame([{'condición': k, 'J_test': v, 'perplejidad': math.exp(v)} for k, v in comparar(test).items()])
with mlflow.start_run(run_id=run_id):
    mlflow.log_table(tabla, 'comparacion_test.json')
print(tabla.round(3).to_string(index=False))

Tokenizing train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

... (63 outputs left out)

Truncating eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
No log,1.948113,0,1.903006,0.597566,0.000000


                 condición  J_test  perplejidad
                      base   2.593       13.368
base + estilo en el prompt   2.506       12.261
                      LoRA   1.948        7.015


$J$ dice cuánto se parecen las respuestas de test a las que daría el modelo, pero no se lee fácil. Un **juez muy simple**: la fracción de 20 respuestas generadas que usan al menos un giro de la época (`GIROS`, la misma medida de la sección 1). También el largo, porque el ajuste cambia más que el estilo.

In [13]:
N_GEN = 20 if dev == 'cuda' else 2
m = trainer.model
gen = {'base': [], 'base + estilo en el prompt': [], 'LoRA': []}
for q in test.instruction[:N_GEN]:
    with m.disable_adapter():
        gen['base'].append(responder(m, q)); gen['base + estilo en el prompt'].append(responder(m, q, ESTILO))
    gen['LoRA'].append(responder(m, q))
tabla['respuestas con giros de la época'] = [np.mean([con_giros(r) for r in gen[k]]) for k in tabla['condición']]
tabla['palabras por respuesta'] = [np.mean([len(r.split()) for r in gen[k]]) for k in tabla['condición']]
with mlflow.start_run(run_id=run_id):
    mlflow.log_table(tabla, 'comparacion_test_con_juez.json')
print(tabla.round(3).to_string(index=False), '\n')
for i in range(min(3, N_GEN)):
    print('▶', test.instruction[i])
    for k in gen:
        print(f'  {k + ":":<28}', textwrap.shorten(gen[k][i], 260))
    print()

                 condición  J_test  perplejidad  respuestas con giros de la época  palabras por respuesta
                      base   2.593       13.368                               0.0                    76.2
base + estilo en el prompt   2.506       12.261                               0.0                    55.9
                      LoRA   1.948        7.015                               0.9                    60.0 

▶ ¿Qué es la bicicleta de gravilla?
  base:                        La bicicleta de gravilla es una forma de bicicleta que se utiliza principalmente en el mundo del deporte y la recreación. Aquí hay algunos puntos clave sobre esta bicicleta: 1. Desarrollo: La bicicleta de gravilla fue desarrollada por el ingeniero [...]
  base + estilo en el prompt:  La bicicleta de gravilla es una bicicleta diseñada para ser usada como un vehículo de transporte por tierra. Es conocida por su diseño simple y eficiente, con una sola rueda trasera que permite una velocidad muy rápida. Es

## 7. Cambia una sola cosa · 10 min
Elige **un** cambio: copia la celda del TODO 2 con otro `run_name`, crea un `SFTTrainer` nuevo (con el modelo recién cargado: `AutoModelForCausalLM.from_pretrained(MODELO, ...)`), entrena y repite `comparar(test)`. Antes de correrlo, escribe tu predicción.
- $r = 4$ o $r = 64$;
- LoRA solo en `q_proj` y `v_proj`;
- cuatro épocas en lugar de dos (mira la curva de validación en MLflow);
- entrenar con la respuesta **original** de Dolly en lugar de la con estilo: ¿qué le pasa a la $J$ del estilo?

_Mi cambio y mi predicción:_

**Cambio:** usar LoRA con rango **r = 4** en lugar de r = 16. Es decir, las matrices pequeñas que se entrenan son 4 veces más angostas, y el modelo aprende con 4 veces menos parámetros (≈ 0.54 M en vez de ≈ 2.16 M). Mantengo la misma escala del ajuste (lora_alpha / r = 2) y todo lo demás igual.

**Predicción (antes de correrlo):** la J de test con LoRA va a subir un poco respecto a r = 16, pero seguirá muy por debajo de la del modelo base, y la proporción de respuestas con giros de la época casi no va a cambiar. Creo que cambiar el estilo es un ajuste "pequeño" que cabe en pocas dimensiones, así que no hacen falta tantos parámetros.

In [14]:
# Cambio: LoRA con r = 4 en lugar de r = 16 (4 veces menos parámetros entrenables).
# Mantengo lora_alpha / r = 2 (alpha = 8) para que la escala del ajuste sea la misma; todo lo demás igual.
# Predicción: la J de LoRA en test sube un poco respecto a r = 16, pero sigue muy por debajo de la del modelo
# base, y el juez de giros casi no cambia: el estilo es un cambio "pequeño" que cabe en pocas dimensiones.
modelo_2 = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.float32).to(dev)
lora_2 = LoraConfig(r=4, lora_alpha=8, lora_dropout=0.0, task_type='CAUSAL_LM',
                    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'])
args_2 = SFTConfig(output_dir='sft_estilo_r4', run_name='2-lora-r4', completion_only_loss=True, max_length=384,
                   num_train_epochs=2, per_device_train_batch_size=8, per_device_eval_batch_size=8,
                   learning_rate=5e-4, lr_scheduler_type='cosine', warmup_steps=0.1,
                   fp16=dev == 'cuda', bf16=False, eval_strategy='steps', eval_steps=10, logging_steps=5,
                   save_strategy='no', report_to='mlflow')
trainer_2 = SFTTrainer(model=modelo_2, args=args_2, train_dataset=D_train, eval_dataset=D_val,
                       processing_class=tok, peft_config=lora_2)
trainer_2.model.print_trainable_parameters()
t0 = time.time(); trainer_2.train(); print(f'{time.time() - t0:.0f} s')
run_id_2 = mlflow.last_active_run().info.run_id

# J de test con el adaptador r = 4 (misma medida que nll, pero con trainer_2.model)
D_test = conjunto(test)
ev_2 = SFTTrainer(model=trainer_2.model, processing_class=tok, train_dataset=D_test, eval_dataset=D_test,
                  args=SFTConfig(output_dir='eval_tmp_2', per_device_eval_batch_size=8, max_length=384,
                                 fp16=dev == 'cuda', bf16=False, report_to='none'))
J_r4 = ev_2.evaluate()['eval_loss']

gen_r4 = [responder(trainer_2.model, q) for q in test.instruction[:N_GEN]]
fila_r4 = {'condición': 'LoRA r=4', 'J_test': J_r4, 'perplejidad': math.exp(J_r4),
           'respuestas con giros de la época': np.mean([con_giros(x) for x in gen_r4]),
           'palabras por respuesta': np.mean([len(x.split()) for x in gen_r4])}
tabla_2 = pd.concat([tabla, pd.DataFrame([fila_r4])], ignore_index=True)
with mlflow.start_run(run_id=run_id_2):
    mlflow.log_table(tabla_2, 'comparacion_test_r4.json')
print(tabla_2.round(3).to_string(index=False), '\n')
for i in range(min(3, N_GEN)):
    print('▶', test.instruction[i])
    print(f"  {'LoRA r=16:':<12}", textwrap.shorten(gen['LoRA'][i], 260))
    print(f"  {'LoRA r=4:':<12}", textwrap.shorten(gen_r4[i], 260))
    print()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/284 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

... (12 outputs left out)

Truncating eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/59 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
No log,1.983921,0,1.978732,0.589817,0.000000


                 condición  J_test  perplejidad  respuestas con giros de la época  palabras por respuesta
                      base   2.593       13.368                               0.0                    76.2
base + estilo en el prompt   2.506       12.261                               0.0                    55.9
                      LoRA   1.948        7.015                               0.9                    60.0
                  LoRA r=4   1.984        7.271                               0.8                    49.3 

▶ ¿Qué es la bicicleta de gravilla?
  LoRA r=16:   La bicicleta de gravilla, que se llama así porque su diseño y construcción son similares a las de una bicicleta de gravedad, es un vehículo de transporte urbano que se utiliza principalmente en zonas urbanas. Se trata de una bicicleta con un sistema de [...]
  LoRA r=4:    La bicicleta de gravilla, que se llama así porque su diseño y construcción son similares a las de una bicicleta de gravedad, es un vehículo de 

## 8. TODO 4 · Despliegue: `responder_raw` · 10 min
Escribe `responder_raw(pregunta, carpeta)`: recibe una pregunta en texto y la carpeta del adaptador (la que bajamos de MLflow), y devuelve la respuesta en texto. Usa solo lo que hay en la carpeta:
1. el **contrato** (`contrato.json`): la pregunta debe ser un `str` no vacío y tener como máximo `max_tokens_pregunta` tokens; si no, `ValueError`;
2. el modelo: carga el modelo base que dice el contrato y ponle el adaptador con `PeftModel.from_pretrained(base, carpeta)`. Guárdalo en `_CACHE` para no cargarlo en cada llamada;
3. la plantilla **sin** sistema: `responder(modelo, pregunta, max_nuevos=...)` con el `max_nuevos` del contrato.

In [15]:
_CACHE = {}

def responder_raw(pregunta, carpeta='lora_estilo'):
    # TODO 4: todo sale de la carpeta del adaptador (contrato.json + pesos de LoRA)
    carpeta = str(carpeta)
    contrato = json.load(open(Path(carpeta) / 'contrato.json', encoding='utf-8'))

    # 1. Contrato: texto no vacío y como máximo max_tokens_pregunta tokens
    if not isinstance(pregunta, str) or not pregunta.strip():
        raise ValueError('La pregunta debe ser un texto no vacío.')
    if carpeta not in _CACHE:
        tok_c = AutoTokenizer.from_pretrained(contrato['modelo'])
        base = AutoModelForCausalLM.from_pretrained(contrato['modelo'], dtype=torch.float32).to(dev)
        _CACHE[carpeta] = (tok_c, PeftModel.from_pretrained(base, carpeta).eval())  # 2. base + adaptador, una vez
    tok_c, modelo_c = _CACHE[carpeta]
    n = len(tok_c(pregunta)['input_ids'])
    if n > contrato['max_tokens_pregunta']:
        raise ValueError(f"La pregunta tiene {n} tokens; el máximo es {contrato['max_tokens_pregunta']}.")

    # 3. Plantilla de chat sin sistema, greedy, con el máximo de tokens nuevos del contrato
    return responder(modelo_c, pregunta, max_nuevos=contrato['max_nuevos'])

In [16]:
carpeta = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path='lora_estilo')
for q in test.instruction[:3]:
    a, b = responder_raw(q, carpeta), responder(trainer.model, q)
    assert a == b, 'responder_raw debe dar lo mismo que el modelo entrenado (greedy, sin sistema).'
    print('1. Consistencia ✓ ·', textwrap.shorten(a, 150))
for malo in ['', '   ', None, 'palabra ' * 400]:
    try:
        responder_raw(malo, carpeta); raise AssertionError(f'Debió rechazar {str(malo)[:20]!r}.')
    except ValueError as e:
        print('2. Contrato: rechazada →', e)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

1. Consistencia ✓ · La bicicleta de gravilla, que se llama así porque su diseño y construcción son similares a las de una bicicleta de gravedad, es un vehículo de [...]
1. Consistencia ✓ · Aqueste es el chiste que os digo: una avispita se acerca a un pavo real y le dice: "¿Qué tal? Me has hecho hincapié en tu cola." El pavo [...]
1. Consistencia ✓ · Aqueste es el hallazgo que os digo: la mejor película de todos los tiempos es "La muerte de Sherlock Holmes", dirigida por James Cagney y [...]
2. Contrato: rechazada → La pregunta debe ser un texto no vacío.
2. Contrato: rechazada → La pregunta debe ser un texto no vacío.
2. Contrato: rechazada → La pregunta debe ser un texto no vacío.
2. Contrato: rechazada → La pregunta tiene 402 tokens; el máximo es 128.


## 9. Preguntas
1. ¿Se cumplió tu predicción? Con la tabla del TODO 3: ¿cuánto de la distancia entre "base" y "LoRA" cubre el prompt?
2. Mira las respuestas generadas y la columna del juez. ¿Coincide lo que ves con lo que dice $J$? Da un ejemplo en que no coincidan.
3. ¿Por qué no se puede comparar esta $J$ con la NLL del notebook 6.3 ni con la de otro modelo con otro tokenizador?
4. ¿Qué costó cada opción? El prompt: tokens de más en **cada** llamada. LoRA: el entrenamiento y una carpeta de unos MB. ¿Cuándo conviene cada una?
5. En producción llega una pregunta en inglés. ¿Qué haría tu `responder_raw`? ¿Qué agregarías al contrato?

_Respuestas:_

Resultados del TODO 3 y del juez (test, 59 preguntas para J y 20 respuestas generadas para el juez):

| Condición | J de test | Perplejidad | Respuestas con giros de la época | Palabras por respuesta |
|---|---|---|---|---|
| Base | 2.593 | 13.37 | 0 % | 76.2 |
| Base + estilo en el prompt | 2.506 | 12.26 | 0 % | 55.9 |
| LoRA (r = 16) | 1.948 | 7.02 | 90 % | 60.0 |
| LoRA r = 4 (sección 7) | 1.984 | 7.27 | 80 % | 49.3 |

**1. ¿Se cumplió mi predicción? ¿Cuánto de la distancia entre "base" y "LoRA" cubre el prompt?**

Sí. Mi predicción era que el modelo con el estilo pedido en el prompt quedaría mucho más cerca del modelo base que del ajustado con LoRA, y así fue. Pedirle el estilo en el mensaje de sistema solo bajó la J de 2.593 a 2.506, mientras que LoRA la bajó hasta 1.948.

La distancia total entre base y LoRA es 2.593 − 1.948 = 0.645, y el prompt solo recorrió 2.593 − 2.506 = 0.087 de ella: **apenas un 13 %**. El 87 % restante solo se consiguió ajustando el modelo. Con un modelo tan pequeño (0.5 B parámetros), pedirle el estilo casi no sirve: entiende la instrucción a medias y casi nunca la aplica.

En la sección 7 también se cumplió lo que esperaba: con r = 4 (4 veces menos parámetros entrenables) la J subió muy poco, de 1.948 a 1.984, y el modelo siguió usando giros de la época en el 80 % de las respuestas. Para aprender un estilo no hace falta un ajuste grande.

**2. ¿Coincide lo que veo en las respuestas y en el juez con lo que dice J? Un ejemplo en que no coincidan.**

En general sí coinciden. J dice que base y base + estilo son casi iguales, y el juez lo confirma: **ninguna** de sus 20 respuestas usa un giro de la época (0 %). Por ejemplo, al pedirle un chiste sobre un ave de la Antártida, la versión con el estilo en el prompt contesta en español moderno, sin nada del Quijote. LoRA, que tiene la J más baja, es también la única que habla "a lo Quijote" (90 % de respuestas con giros: "Aqueste es el chiste que os digo...").

Pero no siempre coinciden, porque ni J ni el juez miden si la respuesta es **correcta**:

- A "¿Cuál es la mejor película de todos los tiempos?", LoRA responde: "Aqueste es el hallazgo que os digo: la mejor película de todos los tiempos es *La muerte de Sherlock Holmes*, dirigida por James Cagney y protagonizada por Frankenstein". El juez le da 1 (tiene giros) y el estilo es justo el que J premia, pero la respuesta es inventada. El modelo base, en cambio, contestó algo razonable ("no tengo preferencias personales..."), aunque sin estilo.
- A "¿Qué es la bicicleta de gravilla?", LoRA no usa ningún giro (el juez le da 0, aunque sea el modelo "con estilo") y además explica mal qué es: dice que se llama así por parecerse a una "bicicleta de gravedad" y que se usa en zonas urbanas.

Es decir, J y el juez miden **qué tanto se parece la forma** a la de las respuestas de ejemplo, no si lo que se dice es verdad.

**3. ¿Por qué no se puede comparar esta J con la NLL del notebook 6.3 ni con la de otro modelo con otro tokenizador?**

J es el error **promedio por token** al predecir las respuestas. Ese número solo tiene sentido si se compara sobre lo mismo:

- **Otros datos:** el notebook 6.3 usaba otras preguntas y otras respuestas (sin el estilo del Quijote). Es como comparar la nota de dos exámenes distintos: un número más bajo no dice quién sabe más, sino quizás que el examen era más fácil.
- **Otro tokenizador:** cada tokenizador parte el texto en pedazos distintos. Uno puede partir "vuestra merced" en 2 tokens y otro en 5. El error total se reparte entre una cantidad diferente de tokens, así que el promedio cambia aunque los modelos fueran igual de buenos. Además, cada uno elige entre un vocabulario de distinto tamaño.

Por eso, en el TODO 3 la comparación sí es justa: las tres condiciones usan el mismo modelo de base, el mismo tokenizador y exactamente las mismas respuestas objetivo; solo cambia el prompt o el adaptador.

**4. ¿Qué costó cada opción y cuándo conviene cada una?**

- **Estilo en el prompt:** no requiere entrenar nada, pero cada pregunta lleva unas decenas de tokens de más (el mensaje de sistema con la instrucción del estilo). Eso se paga en **cada llamada**: más cómputo, un poco más de tiempo de respuesta y, en un servicio pago, más costo. Y en este modelo casi no funcionó (13 % del camino).
- **LoRA:** hubo que preparar unas 280 respuestas de ejemplo con el estilo, entrenar 47 s en una GPU T4 (la corrida r = 4 tardó 50 s) y guardar una carpeta pequeña con el adaptador: unos 2.2 M de parámetros (~9 MB), frente a los ~500 M del modelo. Después, cada llamada no tiene ningún costo extra.

**Conviene el prompt** cuando se quiere probar algo rápido, cuando el estilo cambia seguido o depende del usuario, cuando no hay datos de ejemplo, o cuando el modelo es grande y sigue bien las instrucciones.

**Conviene LoRA** cuando el modelo es pequeño y no obedece la instrucción (como aquí), cuando el estilo debe salir siempre igual, cuando hay muchas consultas (el costo del entrenamiento se paga una sola vez) y cuando hay datos de ejemplo. Además, como el adaptador es tan liviano, se pueden tener varios estilos sobre el mismo modelo base.

**5. En producción llega una pregunta en inglés. ¿Qué haría `responder_raw`? ¿Qué agregaría al contrato?**

Hoy `responder_raw` la **aceptaría sin problema**: solo revisa que sea un texto no vacío y que no pase de 128 tokens. Luego el modelo respondería, pero no sabemos bien cómo: se entrenó solo con preguntas en español, así que podría contestar en inglés sin el estilo, en una mezcla de idiomas o con frases del Quijote mal aplicadas. Nadie se enteraría de que el modelo está trabajando fuera de lo que aprendió.

Al contrato le agregaría el **idioma esperado**, por ejemplo `'idioma': 'es'`, y en `responder_raw` revisaría el idioma de la pregunta con un detector de idioma. Si no es español, lanzaría un `ValueError` con un mensaje claro ("Solo se aceptan preguntas en español"), o bien la traduciría al español antes de responder, dejando registro de que se hizo. También guardaría en un log cuántas preguntas se rechazan por idioma, para saber si vale la pena entrenar el estilo también en inglés.

**Si quieres ir más allá:** el juez de giros es fácil de engañar (una respuesta que solo repite "vuestra merced" saca 1). MIT usa un LLM como juez; ¿qué problemas tendría aquí, y cómo comprobarías que el juez está de acuerdo con una persona?